## Reader mode: CONSTRUCTED monthly factor attribution

The notebook’s default run is a constructed-data method demonstration. All newly calculated values are CONSTRUCTED; they are not market observations or a re-verification of the historical study. French raw, extracted, and processed data are not included. The rights and permitted uses for those data have not been established, so the French analysis is not run here.

The code and labels below describe the constructed analysis shown in this Notebook.

# Project 2: CAPM and Fama–French Factor Regression

By default, this notebook estimates CAPM, FF3, and FF5 for one constructed asset. Its input is a 120-month factor series generated with seed 102, not industry market data such as NoDur, HiTec, or Enrgy. The demonstration teaches same-month excess returns, conditional Betas, model intercepts, HAC inference, and residuals.

All three specifications use the same constructed factor columns, so the models in this constructed demonstration are strictly nested. The results are a method demonstration, not a causal effect, future return, or trading advantage. A separate historical report preserves nine regressions for three industries and three models on the 2010–2019 French snapshot. That report has a different data and evidence identity; it is not new output from this notebook.

This project uses the Capital Asset Pricing Model (CAPM), the Fama–French Three-Factor Model (FF3), and the Fama–French Five-Factor Model (FF5). Ordinary Least Squares (OLS) estimates coefficients; Heteroskedasticity and Autocorrelation Consistent (HAC) covariance estimation is used for inference; confidence intervals (CIs) describe pointwise uncertainty. The theory and worked calculations are in Chapters 8 and 15.

See the [project README](../README.md), [Reader Guide](../READER_GUIDE.md), [historical research report](../research_report/report.md), [exercises](../exercises.md), [solutions](../solutions.md), and [data dictionary](../data_dictionary.md).

## Constructed sample, equation, and timing

The monthly index starts in 2010-01 and contains 120 months. The random seed is 102; five factor columns are generated as pseudo-random draws from N(0, 0.03²), and RF is fixed at 0.001. The single constructed asset’s total return is:

\[
R_t=RF_t+0.002+1.1\,f_{\mathrm{MKT},t}+0.4\,f_{\mathrm{SMB},t}+\epsilon_t,\qquad
\epsilon_t\sim N(0,0.015^2).
\]

The excess-return target is constructed asset total return less same-month RF once: \(y_t=asset_t-RF_t\). These are decimal monthly returns, not French observations.

- CAPM uses Mkt-RF; FF3 uses Mkt-RF, SMB, HML; FF5 uses Mkt-RF, SMB, HML, RMW, CMA. All three select columns from the same constructed factor matrix, so these models are strictly nested in this run.
- Here, names such as Mkt-RF are constructed column labels; they do not mean that official French factors were used. RF is subtracted once from the asset’s total return.
- All three main models use HAC 6. Only FF3 is also estimated with HAC 3, 6, and 12; changing the bandwidth changes standard errors, intervals, and p-values, not OLS point estimates.
- The code checks that every model has 120 observations and verifies excess return = fitted value + residual. The notebook does not calculate or assert a sample-variance identity.
- These months are only the constructed array index; they are not historical announcement, signal, or execution times. The notebook does not run a chronological forecast or trading test.

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project02_factor_regression_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from quant_course.factor_models import fit_monthly_factor_model
print("CONSTRUCTED factor attribution; French snapshot/alignment verification SKIPPED: rights unknown")


In [ ]:
rng = np.random.default_rng(102)
idx = pd.period_range("2010-01", periods=120, freq="M", name="month")
factors = pd.DataFrame(rng.normal(0, 0.03, (120,5)), index=idx, columns=["Mkt-RF","SMB","HML","RMW","CMA"])
factors["RF"] = 0.001
asset = factors["RF"] + 0.002 + 1.1*factors["Mkt-RF"] + 0.4*factors["SMB"] + rng.normal(0,0.015,120)
models = {"CAPM": ("Mkt-RF",), "FF3": ("Mkt-RF","SMB","HML"), "FF5": ("Mkt-RF","SMB","HML","RMW","CMA")}
fits = {m:fit_monthly_factor_model(asset,factors,cols,hac_maxlags=6) for m,cols in models.items()}
assert all(f.nobs == 120 for f in fits.values())


## Results for the three constructed models

This cell displays an OLS + monthly HAC(6) coefficient table for each of the three models. Each table contains only the parameters estimated by that model: CAPM has const, Mkt-RF; FF3 has const, Mkt-RF, SMB, HML; FF5 has const, Mkt-RF, SMB, HML, RMW, CMA. These are not the nine historical industry regressions; there is no cross-asset Holm adjustment, and no value should be interpreted as historical Alpha.

The table fields are estimate; standard_error_hac, t_hac, and p_hac for the HAC standard error, t-statistic, and p-value; and ci_low_hac and ci_high_hac for the 95% interval endpoints. const is the model intercept, and factor rows are conditional loadings. Omitted factors have no row; that does not mean their estimated value is zero.

Each model CSV is written with index=False, so the parameter-name index is not saved. Recover row meaning from the filename and fixed row order above. The six CSV columns remain ordered as estimate, standard_error_hac, t_hac, p_hac, ci_low_hac, ci_high_hac. Do not substitute these constructed results for those in the historical report.

In [ ]:
for model, fit in fits.items():
    print(model, "CONSTRUCTED OLS + monthly HAC(6)")
    display(fit.coefficient_table)
    fit.coefficient_table.to_csv(RUN_DIR / (model + "_coefficients_CONSTRUCTED.csv"), index=False)


## HAC bandwidth and sample variance

Changing the HAC bandwidth changes standard errors, intervals, and p-values, not OLS point estimates. For an OLS projection of the same 120 months with ddof=1, the sample-variance identity is: \(\operatorname{Var}(y)=\operatorname{Var}(\hat y)+\operatorname{Var}(e)\). This is an in-sample variance-accounting relationship in theory; this notebook does not calculate or assert the decomposition.

The code actually checks that every fit has 120 observations and uses assert_allclose to verify excess return = fitted_excess + residuals. The reconstruction identity is not a variance-decomposition check. Even when it holds, this in-sample OLS relationship does not show that future risk can be predicted or that correlated factors form independent risk shares.

In [ ]:
for lag in (3, 6, 12):
    f = fit_monthly_factor_model(asset, factors, models["FF3"], hac_maxlags=lag)
    print("HAC bandwidth", lag, "months; R2", f.r_squared)
    display(f.coefficient_table)
fit = fits["FF3"]
np.testing.assert_allclose(asset - factors["RF"], fit.fitted_excess + fit.residuals)


## Reading the plot

This default notebook generates only the plot described below. It does not generate the five research figures in the historical report.

- Constructed FF3 residual plot: the x-axis label is Month, but the code passes residuals.to_numpy() to plot without date coordinates, so Matplotlib uses observation ordinals 0–119, not calendar months. The y-axis label is Decimal excess return; the plotted values are FF3 excess-return residuals in decimal monthly-return units. They are neither asset total returns nor unfitted excess returns.
- The five historical figures—Alpha, factor loadings, variance decomposition, historical residuals, and chronological attribution error—and their axis descriptions are in the [historical research report](../research_report/report.md). They are not outputs of this constructed notebook run.

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(fits["FF3"].residuals.to_numpy())
ax.set(title="CONSTRUCTED FF3 residuals", xlabel="Month", ylabel="Decimal excess return")
fig.tight_layout(); fig.savefig(RUN_DIR / "project02_residuals_CONSTRUCTED.png", dpi=130)
plt.show()


## Sources and interpretation boundaries

These official pages describe the historical data and research background: [Kenneth R. French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), [10 Industry Portfolios](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_10_ind_port.html), [FF3 factor description](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_factors.html), and [FF5 factor description](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html). The original research background is Fama and French (1993) and (2015): [1993](https://www.sciencedirect.com/science/article/pii/0304405X93900235) and [2015](https://www.sciencedirect.com/science/article/pii/S0304405X14002323). The historical market-data files are not included, and these source pages do not grant permission to use or redistribute them.

The SMB construction differs between the historical official FF3 and FF5 files; their local common Mkt-RF and SMB series also differ, so the original historical regressions are not strictly nested. The source documents the SMB construction difference, but the month-level difference has not been reproduced from underlying portfolios; the reason for the Mkt-RF file-value difference is unknown. The current constructed notebook uses the same generated factor columns in all three specifications, so its nesting relationship differs from that of the official historical files.

The HAC bandwidth in this notebook is an inference-covariance choice; it does not fix model omissions or establish a causal market effect. With no transaction ledger, costs, financing, and borrowing are not applicable; this is not a zero-cost analysis. French raw, extracted, and processed data are not included, and the rights and permitted uses have not been established. Publicly accessible pages alone do not authorize execution or redistribution.